# Team 12 – Integrated agent graph (QDC Fall 2026) – v4

```
START → data → macro → technical → portfolio_manager → validate ─┬─ ok ─────────────────→ human_approval → orders → risk → log → END
                                          ↑                       ├─ errors, attempts < 3 ┘ (back to portfolio_manager)
                                          └───────────────────────┘
                                                                  └─ still wrong / AI down → fallback → human_approval
```

| Node | Who | What |
|---|---|---|
| **data** | Morgane | refreshes `data/prices.csv`, quality checks (`data_issues.csv`), metrics (`metrics.csv`) |
| **macro** | Gonçalo | Google News headlines → AI: facts with headline ids + **one net view per theme** (structured) |
| **technical** | Juliet | Python only (no AI): momentum score, trend vs 50-day average, breakout, RSI → **one technical view per ETF and per theme** |
| **portfolio_manager** | Francesco | AI proposes target weights + a reason per ETF + an investment memo (JSON) |
| **validate** | Francesco | Python checks the proposal: our rules (25% / 35% planning margin), macro view, technical view, data quality, and that every number the AI cites is really in its input tables; if wrong, back to the AI (max 3 attempts) |
| **fallback** | Francesco | no valid AI proposal → no trades, the team decides |
| **human_approval** | team | the graph **stops**: the team approves, edits the weights or rejects – nothing is approved by default |
| **orders** | Francesco | `weights_to_orders`: max 5 feasible orders |
| **risk** | Gonçalo | `check_risk_rules`: final check on the hard limits (20% / 10% / 20% / 40% / €500) |
| **log** | Francesco | trade log + full run log (prompts, AI answers, model, who decided and why) in `trading/` |

**Rule of the whole graph:** the AI never computes a number. Every figure comes from Python; the AI only reads them, chooses and explains.

**How to run it on Colab**
1. Upload `prices.csv` and `tickers.csv` (from `data/` on GitHub) and `technical_agent.py` (from `Technical_code/`) with the folder icon on the left. On a **real trading day**, also upload `portfolio.csv` and `trade_log.csv` from the last `trading_logs.zip`. Gemini key in Colab Secrets as `LLM_API_KEY`.
2. Run the cells **one at a time**, from top to bottom (do not use "Run all"). Sections 9a and 11 are tests **without** Gemini.
3. Section 12 runs the real graph and stops at the human approval. Section 13 records the team's decision: nothing is approved by default.
4. Real trading day only: section 13b saves our new portfolio after the orders are sent to Belfius.
5. At the end, download the `trading/` folder (section 14): Colab deletes files when the session ends.

## 0. Install (once per session)

In [ ]:
%pip install -q pandas numpy yfinance requests feedparser langgraph langchain-openai openai python-dotenv

## 1. Project folder and files
On Colab, files you upload go to the top folder: this cell moves `prices.csv` and `tickers.csv` into `data/`, `technical_agent.py` into `Technical_code/`, and `portfolio.csv` and `trade_log.csv` (real trading days) into `trading/`, where the code expects them.

In [ ]:
import os, sys, json, time, math, re, shutil
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
now_paris = lambda: datetime.now(ZoneInfo("Europe/Paris"))     # Colab clock is UTC
import numpy as np
import pandas as pd

for folder in [Path.cwd(), *Path.cwd().parents]:          # project folder = the one with data/prices.csv
    if (folder / "data" / "prices.csv").exists():
        os.chdir(folder); break
Path("data").mkdir(exist_ok=True)
Path("trading").mkdir(exist_ok=True)                        # trade log + run logs + our portfolio
Path("Technical_code").mkdir(exist_ok=True)                 # Juliet's technical_agent.py
for f, dest in [("prices.csv", "data"), ("tickers.csv", "data"), ("technical_agent.py", "Technical_code"),
                ("portfolio.csv", "trading"), ("trade_log.csv", "trading")]:
    if Path(f).exists() and not (Path(dest) / f).exists():  # Colab: move uploaded files into place
        shutil.move(f, Path(dest) / f)

print("Project folder:", Path.cwd())
for f in ["data/prices.csv", "data/tickers.csv", "Technical_code/technical_agent.py", "trading/portfolio.csv", "trading/trade_log.csv"]:
    print(f"{f:<34}", "OK" if Path(f).exists() else ("MISSING" if not f.startswith("trading") else "not uploaded (fine for a dry run)"))

## 2. Settings
### 2a. Competition rules + our balanced risk profile
Names and values are the same as in the `macro_news_risk (revised)` notebook (Gonçalo), so there is **one** set of rules for the whole team.

In [ ]:
# ---- Competition rules - do not change ----
MAX_ORDERS_PER_DAY = 5          # buys and sells both count
MIN_CASH_EUR       = 0.0        # no borrowing
STARTING_CAPITAL   = 10_000.0   # EUR 10,000 per team
DECIMALS           = 4          # fractional units allowed
DIVIDEND_WINDOW    = (5, 23)    # days of December when holding a distributing ETF costs us its dividend (not credited in the competition)

# ---- Our risk rules: BALANCED profile (hard limits, checked by check_risk_rules) ----
MAX_WEIGHT_PER_ETF   = 0.20                                  # 1) max 20% per ETF ...
EXEMPT_FROM_LIMITS   = ["OVERNIGHT CASH"]                    #    ... except money market (behaves like cash)
MAX_LEVERAGED_WEIGHT = 0.10                                  # 2) max 10% in leveraged/inverse ETFs together
MIN_DEFENSIVE_WEIGHT = 0.20                                  # 3) min 20% defensive (incl. uninvested cash)
DEFENSIVE_ETFS       = ["OVERNIGHT CASH", "EUR GOVT BOND", "GLOBAL AGG BOND"]   # confirmed by the team
MAX_THEME_WEIGHT     = 0.40                                  # 4) max 40% per theme
THEME_GROUPS     = {"Frontier tech": "Equity / Tech"}        #    themes that move together count as one
UNDERLYING_THEME = {"NASDAQ 3X": "Equity / Tech", "EURO STOXX 2X": "European equity",       # leveraged ETFs count
                    "COPPER LVG": "Industrial & strategic metals", "PALLADIUM LVG": "Precious metals"}  # as their underlying
MIN_ORDER_EUR        = 500.0                                 # 5) min order; 6) never sell more than we hold

# ---- Planning margin for the Portfolio Manager (agreed in the team) ----
PLAN_MIN_DEFENSIVE = 0.25       # the AI plans with >= 25% defensive (hard limit 20%)
PLAN_MAX_THEME     = 0.35       # the AI plans with <= 35% per theme (hard limit 40%)

# ---- Graph options ----
STRICT_MAX_MOVES   = True       # the AI must propose a plan that needs at most 5 orders today
MAX_PM_ATTEMPTS    = 3          # 1 proposal + max 2 corrections (each = 1 Gemini call)
REFRESH_DATA       = True       # data node downloads prices from Yahoo (False = use data/prices.csv as it is)
DATA_MODE          = "full"     # "full" = whole history again (safest, ~1-2 min) | "update" = only the last days
USE_AI_DATA_REPORT = False      # True = AI data report (several extra Gemini calls: free quota!)
LOG_RUNS           = True       # write trading/trade_log.csv and trading/runs/<run_id>.json
TRADE_LOG_FILE     = Path("trading/trade_log.csv")
RUNS_DIR           = Path("trading/runs")
PORTFOLIO_FILE     = Path("trading/portfolio.csv")           # format: code, units + a CASH row

### 2b. Data settings (Morgane, copied from `Data_code/data_agent.ipynb`)

In [ ]:
DATA_DIR     = Path("data")
TICKERS_FILE = DATA_DIR / "tickers.csv"      # 50 ETFs + Yahoo symbols (8 fixed by hand: never regenerate)
PRICES_FILE  = DATA_DIR / "prices.csv"
METRICS_FILE = DATA_DIR / "metrics.csv"
ISSUES_FILE  = DATA_DIR / "data_issues.csv"
REPORT_FILE  = DATA_DIR / "data_report.md"
START_DATE   = "2021-01-01"

LEVERAGE = {"NASDAQ 3X": 3, "COPPER LVG": 3, "PALLADIUM LVG": 2,
            "EURO STOXX 2X": 2, "S&P 500 SHORT": -1, "STOXX SHORT": -2}
SPLITS = {"EURO BANKS": ("2026-10-26", 80),          # 80:1 split (349.80 -> 4.43)
          "PALLADIUM LVG": ("2022-12-19", 4)}        # 4:1 split
BAD_DATES = ["2025-10-24"]                            # corrupted Yahoo day, removed for all ETFs

TRADING_DAYS, FFILL_LIMIT, LONG_GAP_DAYS = 252, 5, 5
EXTREME_MOVE, SUSPICIOUS_MOVE, SHORT_HISTORY = 0.20, 0.08, 252
EXPECTED_CURRENCY = "EUR"
WINDOWS = {"1w": 5, "1m": 21, "3m": 63, "6m": 126, "1y": 252}

## 3. Gemini (one setup for all agents)
Key from **Colab Secrets** (`LLM_API_KEY`) or from the `.env` file on a computer. Never in the code, never on GitHub.

In [ ]:
try:
    from google.colab import userdata
    os.environ["LLM_API_KEY"] = userdata.get("LLM_API_KEY")
except Exception:
    from dotenv import load_dotenv; load_dotenv()
os.environ.setdefault("LLM_BASE_URL", "https://generativelanguage.googleapis.com/v1beta/openai/")

CANDIDATE_MODELS = ["gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash", "gemini-3.5-flash", "gemini-flash-latest"]
os.environ["LLM_MODEL"] = ""
busy = []                                                     # models that exist but have used up their free quota
if os.environ.get("LLM_API_KEY"):
    from openai import OpenAI
    client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"], timeout=60, max_retries=2)
    for m in CANDIDATE_MODELS:
        try:
            client.chat.completions.create(model=m, messages=[{"role": "user", "content": "Say hi"}])
            os.environ["LLM_MODEL"] = m; break                # first model that really answers
        except Exception as e:
            if "RateLimit" in type(e).__name__:
                busy.append(m)
                print(f"{m}: busy (free quota used up for now), trying the next one")
            else:
                print(f"{m}: not available ({type(e).__name__})")
    if not os.environ["LLM_MODEL"] and busy:                  # nobody answered: take the first busy one, the graph waits and retries
        os.environ["LLM_MODEL"] = busy[0]
        print(f"All models busy: using {busy[0]} (the graph will wait and retry)")
print("Gemini model:", os.environ["LLM_MODEL"] or "none (the graph still runs, without AI)")

def get_llm():
    if not all(os.environ.get(k) for k in ("LLM_BASE_URL", "LLM_MODEL", "LLM_API_KEY")):
        return None
    from langchain_openai import ChatOpenAI
    return ChatOpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"],
                      model=os.environ["LLM_MODEL"], temperature=0, timeout=90, max_retries=2)

## 4. Data agent (Morgane)
### 4a. Download, clean, checks, metrics
Copied from `Data_code/data_agent.ipynb` (parts 3–5). Two additions agreed with the team: today's row is never used (it is an intraday price, not a close), and `update()` downloads the last week again, so a wrong last row is always replaced.

In [ ]:
COLUMNS = ["date", "code", "isin", "yahoo", "theme", "currency", "close", "adj_close", "volume", "leverage", "return"]

def today_paris():
    """Today's date in Brussels/Paris time (Colab runs in UTC)."""
    return pd.Timestamp.now(tz="Europe/Paris").normalize().tz_localize(None)

def get_currencies(symbols) -> dict:
    import yfinance as yf
    out = {}
    for s in symbols:
        try:
            out[s] = yf.Ticker(s).fast_info["currency"]
        except Exception:
            out[s] = "UNKNOWN"
    return out

def download(start: str, currencies: dict | None = None) -> pd.DataFrame:
    """Raw Yahoo data, one row per (date, ETF). One single request for all ETFs."""
    import yfinance as yf
    t = pd.read_csv(TICKERS_FILE).fillna("")
    t = t[t["yahoo"] != ""]
    raw = yf.download(t["yahoo"].tolist(), start=start, auto_adjust=False,
                      group_by="ticker", progress=False, threads=True, timeout=30)
    if raw is None or raw.empty:
        raise RuntimeError("Yahoo returned no data: check the connection or try again in a few minutes")
    currencies = currencies or get_currencies(t["yahoo"])
    rows = []
    for r in t.itertuples():
        if r.yahoo not in raw.columns.get_level_values(0):
            print(f"WARNING: no data for {r.code} ({r.yahoo})"); continue
        df = raw[r.yahoo][["Close", "Adj Close", "Volume"]].dropna(subset=["Close"]).reset_index()
        df.columns = ["date", "close", "adj_close", "volume"]
        df = df.assign(code=r.code, isin=r.isin, yahoo=r.yahoo, theme=r.theme, currency=currencies.get(r.yahoo, "UNKNOWN"))
        rows.append(df)
    return pd.concat(rows, ignore_index=True)

def clean(data: pd.DataFrame) -> pd.DataFrame:
    """Fix known Yahoo errors + add leverage. Only for RAW rows (never twice on the same rows)."""
    data = data.copy()
    data["date"] = pd.to_datetime(data["date"])
    data = data[~data["date"].isin(pd.to_datetime(BAD_DATES))]
    data = data[data["date"] < today_paris()]       # today's row is intraday, not a close: orders use T-1
    for c, (split_date, ratio) in SPLITS.items():
        mask = (data["code"] == c) & (data["date"] < pd.Timestamp(split_date))
        data.loc[mask, ["close", "adj_close"]] /= ratio
    data["leverage"] = data["code"].map(LEVERAGE).fillna(1).astype(int)
    return data

def add_returns(data: pd.DataFrame) -> pd.DataFrame:
    data = data.sort_values(["code", "date"]).reset_index(drop=True)
    data["return"] = data.groupby("code")["adj_close"].pct_change(fill_method=None)
    return data

def full():
    data = add_returns(clean(download(START_DATE)))
    data[COLUMNS].to_csv(PRICES_FILE, index=False)
    print(f"Done: {data['code'].nunique()} ETFs, {data['date'].min():%Y-%m-%d} to {data['date'].max():%Y-%m-%d}")

def update(overlap_days=7):
    """Downloads the last week again + the new days: rows already in the file are replaced by the new download."""
    old = pd.read_csv(PRICES_FILE, parse_dates=["date"])
    if "leverage" not in old.columns:
        raise RuntimeError("Old price file without fixes: run full() once")
    old = old[old["date"] < today_paris()]                                    # never keep an intraday row
    start = (old["date"].max() - pd.Timedelta(days=overlap_days)).strftime("%Y-%m-%d")
    currencies = old.drop_duplicates("yahoo").set_index("yahoo")["currency"].to_dict()
    try:
        new = clean(download(start, currencies))
    except RuntimeError:
        print("No new prices on Yahoo yet. Last date:", f"{old['date'].max():%Y-%m-%d}"); return
    data = add_returns(pd.concat([old.drop(columns="return"), new]).drop_duplicates(["date", "code"], keep="last"))
    data[COLUMNS].to_csv(PRICES_FILE, index=False)
    print(f"Done: up to {data['date'].max():%Y-%m-%d} (last {overlap_days} days downloaded again)")

def load_prices() -> pd.DataFrame:
    df = pd.read_csv(PRICES_FILE, parse_dates=["date"])
    df = df[df["date"] < today_paris()]  # never use today's intraday prices
    if "leverage" not in df.columns:     # old raw file from before the fixes: clean it in memory
        print("Raw prices.csv (no fixes yet): cleaning in memory. Run full() to save a clean file.")
        df = add_returns(clean(df.drop(columns=["return"], errors="ignore")))
        df["currency"] = df.get("currency", "UNKNOWN")
    return df.sort_values(["code", "date"]).reset_index(drop=True)

def detect_issues(df: pd.DataFrame) -> pd.DataFrame:
    issues = []
    add = lambda c, d, kind, detail: issues.append({"code": c, "date": d, "issue": kind, "detail": detail})
    start, end = df["date"].min(), df["date"].max()
    for c, g in df.groupby("code"):
        g = g.set_index("date").sort_index()
        lev, cur = int(g["leverage"].iloc[0]), g["currency"].iloc[0]
        ret = g["adj_close"].pct_change(fill_method=None)
        if cur != EXPECTED_CURRENCY:
            add(c, None, "currency_not_eur", f"currency '{cur}'")
        if (g["close"] <= 0).any():
            add(c, None, "non_positive_price", f"{int((g['close'] <= 0).sum())} rows <= 0")
        if len(g) < SHORT_HISTORY:
            add(c, g.index[0].date(), "short_history", f"only {len(g)} trading days")
        elif g.index[0] > start + pd.Timedelta(days=30):
            add(c, g.index[0].date(), "late_start", "starts later than the other ETFs")
        if g.index[-1] < end:
            add(c, g.index[-1].date(), "stale", f"no price after {g.index[-1]:%Y-%m-%d}")
        missing = ~pd.bdate_range(g.index[0], g.index[-1]).isin(g.index)
        runs = pd.Series(missing).groupby((~pd.Series(missing)).cumsum()).sum()
        if len(runs) and runs.max() > LONG_GAP_DAYS:
            add(c, None, "long_gap", f"longest gap {int(runs.max())} trading days")
        for d, x in ret[ret.abs() > EXTREME_MOVE].items():
            note = f" (leverage {lev:+d}x: may be genuine)" if lev != 1 else ""
            add(c, d.date(), "extreme_move", f"{x:+.1%} in one day{note}")
        nxt = ret.shift(-1)
        bad = (ret.abs() > SUSPICIOUS_MOVE) & (g["volume"] == 0) & (np.sign(nxt) == -np.sign(ret)) & (nxt.abs() > 0.5 * ret.abs())
        for d in ret[bad].index:
            add(c, d.date(), "suspicious_print", f"{ret[d]:+.1%} on zero volume, then {nxt[d]:+.1%}")
    return pd.DataFrame(issues, columns=["code", "date", "issue", "detail"])

def max_drawdown(p: pd.Series) -> float:
    p = p.dropna()
    return float((p / p.cummax() - 1).min()) if len(p) else np.nan

def compute_metrics(df: pd.DataFrame) -> pd.DataFrame:
    adj = df.pivot(index="date", columns="code", values="adj_close").sort_index().ffill(limit=FFILL_LIMIT)
    rets = adj.pct_change(fill_method=None)
    info = df.drop_duplicates("code").set_index("code")
    rows = []
    for c in adj.columns:
        p, r = adj[c].dropna(), rets[c].dropna()
        vol = df.loc[df["code"] == c].set_index("date")["volume"]
        row = {"code": c, "theme": info.loc[c, "theme"], "leverage": int(info.loc[c, "leverage"]),
               "currency": info.loc[c, "currency"], "first_date": p.index[0].date(), "last_date": p.index[-1].date(),
               "history_days": len(p), "last_close": float(df.loc[df["code"] == c, "close"].iloc[-1])}
        for name, n in WINDOWS.items():
            row[f"ret_{name}"] = p.iloc[-1] / p.iloc[-n - 1] - 1 if len(p) > n else np.nan
        for name in ["1m", "3m", "1y"]:
            n = WINDOWS[name]
            row[f"vol_{name}"] = r.iloc[-n:].std() * np.sqrt(TRADING_DAYS) if len(r) >= n else np.nan
        row["max_drawdown_all"] = max_drawdown(p)
        row["max_drawdown_1y"] = max_drawdown(p.iloc[-WINDOWS["1y"]:]) if len(p) > WINDOWS["1y"] else np.nan
        row["current_drawdown"] = p.iloc[-1] / p.max() - 1
        row["avg_volume_1m"] = vol.iloc[-WINDOWS["1m"]:].mean()
        row["zero_volume_days_1m"] = int((vol.iloc[-WINDOWS["1m"]:] == 0).sum())
        rows.append(row)
    return pd.DataFrame(rows).set_index("code")

def build_outputs() -> dict:
    """Checks + metrics, saved for the other agents. Returns a short summary."""
    df = load_prices()
    metrics, issues = compute_metrics(df), detect_issues(df)
    metrics.to_csv(METRICS_FILE)
    issues.to_csv(ISSUES_FILE, index=False)
    return {"n_etfs": int(metrics.shape[0]),
            "date_range": [f"{df['date'].min():%Y-%m-%d}", f"{df['date'].max():%Y-%m-%d}"],
            "issues_by_type": issues["issue"].value_counts().to_dict(),
            "leverage_factors": metrics.loc[metrics["leverage"] != 1, "leverage"].to_dict(),
            "currencies": metrics["currency"].value_counts().to_dict()}

### 4b. Data agent with AI (optional) + `run_data_agent`
Morgane's mini-agent (tools + LangGraph loop) writes a short data report. It costs several Gemini calls, so in the team graph it is **off** by default (`USE_AI_DATA_REPORT = False`): the data node then runs the same pipeline without AI. With `DATA_MODE = "full"` the whole history is downloaded again at every run (safest: dividends and splits are always up to date).

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

_j = lambda x: json.dumps(x, default=str, indent=1)
_r = lambda d: {k: (round(v, 4) if isinstance(v, (float, np.floating)) and not np.isnan(v) else v) for k, v in d.items()}

@tool
def get_data_status() -> str:
    """Is there a price file, what dates does it cover, is it up to date? Call it first."""
    last_bday = (today_paris() - pd.offsets.BDay(1)).date()
    if not PRICES_FILE.exists():
        return _j({"price_file_exists": False, "last_business_day": last_bday})
    df = pd.read_csv(PRICES_FILE, parse_dates=["date"])
    return _j({"price_file_exists": True, "has_fixes": "leverage" in df.columns,
               "first_date": df["date"].min().date(), "last_date": df["date"].max().date(),
               "n_etfs": df["code"].nunique(), "last_business_day": last_bday,
               "up_to_date": df["date"].max().date() >= last_bday})

@tool
def refresh_prices(mode: str) -> str:
    """Download prices. mode='full' if there is no file or it has no fixes, mode='update' if it is only out of date."""
    try:
        full() if mode == "full" else update() if mode == "update" else None
    except Exception as e:
        return f"Download failed: {e}. Continue with the existing file."
    return get_data_status.invoke({})

@tool
def build_dataset() -> str:
    """Run the quality checks and compute the metrics; saves metrics.csv and data_issues.csv for the team."""
    return _j(build_outputs())

@tool
def list_data_issues(issue_type: str = "all") -> str:
    """Detected issues. issue_type: all, extreme_move, suspicious_print, long_gap, short_history, late_start,
    stale, non_positive_price, currency_not_eur."""
    i = pd.read_csv(ISSUES_FILE)
    i = i if issue_type == "all" else i[i["issue"] == issue_type]
    return _j(i.to_dict(orient="records")) if len(i) else "No issues of this type."

@tool
def get_risk_overview(sort_by: str = "vol_1y", top_n: int = 8) -> str:
    """Rank ETFs by a metric (vol_1y, max_drawdown_all, ret_1m, ret_3m, current_drawdown...)."""
    m = pd.read_csv(METRICS_FILE, index_col="code")
    if sort_by not in m.columns:
        return f"Unknown metric. Available: {', '.join(m.columns)}"
    top = m.sort_values(sort_by, ascending="drawdown" in sort_by).head(top_n)
    cols = list(dict.fromkeys(["theme", "leverage", sort_by, "vol_1y", "max_drawdown_all"]))
    return _j({c: _r(r) for c, r in top[cols].to_dict(orient="index").items()})

@tool
def get_etf_metrics(code: str) -> str:
    """All metrics and issues of one ETF, by its code (e.g. 'WORLD', 'NASDAQ 3X')."""
    m = pd.read_csv(METRICS_FILE, index_col="code")
    if code not in m.index:
        return f"Unknown code. Available: {', '.join(m.index)}"
    i = pd.read_csv(ISSUES_FILE)
    return _j({"metrics": _r(m.loc[code].to_dict()), "issues": i[i["code"] == code].to_dict(orient="records")})

data_tools = [get_data_status, refresh_prices, build_dataset, list_data_issues, get_risk_overview, get_etf_metrics]

DATA_SYSTEM_PROMPT = """You are the Data agent of an ETF portfolio team. You run before the Technical, Macro/News, Risk and
Portfolio Manager agents, who read data/prices.csv and data/metrics.csv.

1. Call get_data_status. No file or has_fixes=false -> refresh_prices('full'). Not up to date -> refresh_prices('update').
   If a download fails, say so and continue with the existing file.
2. Call build_dataset.
3. Review the issues with list_data_issues: for each extreme_move or suspicious_print, say if it looks like a data error
   or a genuine move (leveraged/inverse ETFs can move a lot). Mention any ETF not in EUR.
4. Use get_risk_overview to name the most volatile ETFs and the deepest drawdowns.
5. Write a short data report (max 15 lines): date range, number of ETFs, issues to watch, ETFs with short history,
   leveraged/inverse ETFs and their factor, highest-risk ETFs.

Rules: every number must come from a tool result. Never estimate or invent figures or dates.
You do not recommend trades: that is the Portfolio Manager's job."""

def data_agent_node(state: MessagesState):
    return {"messages": [get_llm().bind_tools(data_tools).invoke([SystemMessage(DATA_SYSTEM_PROMPT)] + state["messages"])]}

_dg = StateGraph(MessagesState)
_dg.add_node("agent", data_agent_node)
_dg.add_node("tools", ToolNode(data_tools))
_dg.add_edge(START, "agent")
_dg.add_conditional_edges("agent", tools_condition, {"tools": "tools", "__end__": END})
_dg.add_edge("tools", "agent")
data_agent_app = _dg.compile()

def run_data_agent(use_ai=False) -> str:
    """Refreshes prices + checks + metrics. With use_ai=True, the AI agent also writes a data report."""
    if not use_ai or get_llm() is None:
        status = json.loads(get_data_status.invoke({}))
        if DATA_MODE == "full" or not status["price_file_exists"] or not status["has_fixes"]:
            full()
        else:
            update()
        report = "[no AI data report] " + json.dumps(build_outputs(), default=str)
    else:
        out = data_agent_app.invoke({"messages": [{"role": "user", "content": "Prepare today's market data and write the data report."}]},
                                    {"recursion_limit": 30})
        report = out["messages"][-1].content
        for m in out["messages"]:
            for call in getattr(m, "tool_calls", None) or []:
                print("   data tool:", call["name"], call["args"])
    REPORT_FILE.write_text(report, encoding="utf-8")
    return report

## 5. Market data in memory + the table for the Portfolio Manager
`load_market_data()` (re)loads prices, metrics and data issues. It runs once here and again inside the **data** node after a refresh, so every agent always uses the same, fresh numbers.

The Portfolio Manager's table comes from Morgane's `metrics.csv`, plus:
- `vs_ma50_%`: price vs its 50-day average (same value as in Juliet's technical table);
- `data_flag`: "check data" if `data_issues.csv` lists a recent problem for that ETF (the AI may not buy it);
- `exdiv_window`: past ex-dividend dates between 5 and 23 December, with the price drop. **Dividends are not credited in the competition**, so holding a distributing ETF on that day costs us the dividend. Computed from the jump of `adj_close / close` on the ex-dividend day.

`prices_warning()` says loudly if the prices are not from the last business day (e.g. a failed download): orders are executed at the T-1 close, so old prices would give wrong orders.

In [ ]:
def etf_theme(code):
    """Theme used for the 40% cap: the underlying exposure, related themes merged (risk rules, Gonçalo)."""
    t = UNDERLYING_THEME.get(code, THEME_OF.get(code))
    return THEME_GROUPS.get(t, t)

RECENT_DAYS = 182                                             # issues older than ~6 months don't block an ETF
ALWAYS_BLOCKING = {"stale", "non_positive_price"}
RECENT_BLOCKING = {"suspicious_print", "extreme_move"}         # extreme moves only for non-leveraged ETFs

def data_flag(code, lev):
    cutoff = prices["date"].max() - pd.Timedelta(days=RECENT_DAYS)
    found = []
    for r in issues[issues["code"] == code].itertuples():
        if r.issue in ALWAYS_BLOCKING:
            found.append(r.issue)
        elif r.issue == "currency_not_eur" and "UNKNOWN" not in str(r.detail):   # UNKNOWN = old file, not an error
            found.append("not EUR")
        elif r.issue in RECENT_BLOCKING and pd.notna(r.date) and r.date >= cutoff:
            if r.issue == "suspicious_print" or lev == 1:
                found.append(f"{r.issue} {r.date:%d/%m/%y}")
    if found:
        return "check data (" + ", ".join(sorted(set(found))) + ")"
    return "short history" if (issues[issues["code"] == code]["issue"] == "short_history").any() else "ok"

def dividend_calendar(df, years=3):
    """Past ex-dividend dates inside the December window, per ETF (from the jump of adj_close/close)."""
    cal, first = {}, df["date"].max() - pd.DateOffset(years=years)
    for c, g in df.sort_values("date").groupby("code"):
        ratio = (g["adj_close"] / g["close"]).to_numpy()
        dates = g["date"].to_numpy()
        step = ratio[:-1] / ratio[1:] - 1                    # < 0: dividend paid, ex-dividend day = the next date
        hits = [(pd.Timestamp(dates[i + 1]), step[i]) for i in range(len(step))
                if step[i] < -0.0005 and pd.Timestamp(dates[i + 1]) >= first]
        window = [(d, x) for d, x in hits if d.month == 12 and DIVIDEND_WINDOW[0] <= d.day <= DIVIDEND_WINDOW[1]]
        cal[c] = "; ".join(f"{d:%d %b %Y} {x:+.1%}" for d, x in sorted(window, reverse=True)) or "-"
    return cal

def prices_warning(prices_date):
    """Text of a warning if the prices are not from the last business day, else None."""
    last, expected = pd.Timestamp(prices_date), today_paris() - pd.offsets.BDay(1)
    if last >= expected:
        return None
    return (f"!! WARNING: prices are from {last:%a %d/%m/%Y}, not from the last business day ({expected:%a %d/%m/%Y}). "
            "Orders are executed at the T-1 close: refresh the data before deciding.")

def build_pm_table():
    m = metrics
    adj = prices.pivot_table(index="date", columns="code", values="adj_close").sort_index()
    vs_ma50 = {c: adj[c].dropna().iloc[-1] / adj[c].dropna().tail(50).mean() - 1 for c in adj.columns}
    divs = dividend_calendar(prices)
    pct = lambda col: (m[col] * 100).round(1).values
    t = pd.DataFrame({
        "code": m.index,
        "theme": m["theme"].values,
        "exposure_theme": [etf_theme(c) for c in m.index],           # theme for the 40% cap
        "leverage": m["leverage"].astype(int).values,
        "defensive": [c in DEFENSIVE_ETFS for c in m.index],
        "ret_1m_%": pct("ret_1m"), "ret_3m_%": pct("ret_3m"), "ret_6m_%": pct("ret_6m"),
        "vol_3m_%": pct("vol_3m"), "max_dd_1y_%": pct("max_drawdown_1y"), "current_dd_%": pct("current_drawdown"),
        "vs_ma50_%": [round(vs_ma50[c] * 100, 1) for c in m.index],
        "zero_vol_days_1m": m["zero_volume_days_1m"].astype(int).values,
        "exdiv_window": [divs.get(c, "-") for c in m.index],
        "data_flag": [data_flag(c, int(m.loc[c, "leverage"])) for c in m.index],
    })
    return t

def load_market_data(verbose=True):
    """(Re)load prices, metrics and issues into memory (used by every agent)."""
    global prices, universe, last_close, PRICES, THEME_OF, metrics, issues, PM_TABLE
    summary = build_outputs()                                   # checks + metrics -> csv files
    prices = load_prices()
    universe = prices.groupby("code").agg(theme=("theme", "last"), leverage=("leverage", "last")).reset_index()
    last_close = prices.sort_values("date").groupby("code")["close"].last()   # T-1 raw close = execution price
    PRICES = last_close.to_dict()
    THEME_OF = universe.set_index("code")["theme"].to_dict()
    metrics = pd.read_csv(METRICS_FILE, index_col="code")
    issues = pd.read_csv(ISSUES_FILE, parse_dates=["date"])
    PM_TABLE = build_pm_table()
    if verbose:
        print(f"{summary['n_etfs']} ETFs, prices {summary['date_range'][0]} to {summary['date_range'][1]}")
        print(prices_warning(summary["date_range"][1]) or "Prices are from the last business day: OK")
        print("Data issues:", summary["issues_by_type"])
        flagged = PM_TABLE[PM_TABLE["data_flag"] != "ok"][["code", "data_flag"]]
        print("ETFs with a data flag:\n" + (flagged.to_string(index=False) if len(flagged) else "none"))
        divs = PM_TABLE[PM_TABLE["exdiv_window"] != "-"][["code", "exdiv_window"]]
        print("Past ex-dividend dates in the December window (dividends NOT credited):\n" + divs.to_string(index=False))
    return summary

_ = load_market_data()

## 6. Portfolio format and orders (Francesco)
One format for everyone: `{"cash": 10000.0, "units": {"WORLD": 15.2}}`. `to_goncalo_portfolio` converts it to the risk table (`code`, `units` + a `CASH` row). `portfolio_after` / `save_portfolio` keep our real portfolio from one trading day to the next.

In [ ]:
def new_portfolio():
    return {"cash": STARTING_CAPITAL, "units": {}}

def to_goncalo_portfolio(pf):
    rows = [{"code": "CASH", "units": pf["cash"]}] + [{"code": c, "units": u} for c, u in pf["units"].items()]
    return pd.DataFrame(rows)

def from_goncalo_portfolio(df):
    return {"cash": float(df.loc[df["code"] == "CASH", "units"].sum()),
            "units": {r.code: float(r.units) for r in df.itertuples() if r.code != "CASH" and r.units > 0}}

def load_portfolio():
    """Current portfolio from trading/portfolio.csv, or all cash if the file doesn't exist yet."""
    if PORTFOLIO_FILE.exists():
        return pd.read_csv(PORTFOLIO_FILE)
    return pd.DataFrame({"code": ["CASH"], "units": [STARTING_CAPITAL]})

def portfolio_value(pf, prices=None):
    prices = PRICES if prices is None else prices
    return pf["cash"] + sum(u * prices[c] for c, u in pf["units"].items())

def current_weights(pf, prices=None):
    prices = PRICES if prices is None else prices
    total = portfolio_value(pf, prices)
    return {c: u * prices[c] / total for c, u in pf["units"].items()}

def check_weights(target_weights, prices=None):
    prices = PRICES if prices is None else prices
    errors = []
    for c, w in target_weights.items():
        if c not in prices: errors.append(f"{c}: not in the ETF list")
        if w < 0:           errors.append(f"{c}: negative weight ({w})")
    if sum(target_weights.values()) > 1.0001:
        errors.append(f"weights sum to {sum(target_weights.values()):.4f} > 1 (no borrowing)")
    return errors

def weights_to_orders(pf, target_weights, prices=None, max_orders=MAX_ORDERS_PER_DAY, min_trade_eur=MIN_ORDER_EUR):
    """Target weights -> at most 5 feasible orders (sells first, biggest moves first, every buy funded)."""
    prices = PRICES if prices is None else prices
    errors = check_weights(target_weights, prices)
    if errors:
        raise ValueError("Invalid weights:\n- " + "\n- ".join(errors))
    total = portfolio_value(pf, prices)
    gaps = []
    etfs = list(target_weights) + sorted(c for c in pf["units"] if c not in target_weights)   # fixed order: reproducible
    for c in etfs:
        gap = target_weights.get(c, 0.0) * total - pf["units"].get(c, 0.0) * prices[c]
        if abs(gap) >= min_trade_eur:
            gaps.append((c, gap))
    gaps.sort(key=lambda x: abs(x[1]), reverse=True)
    sells_left = [g for g in gaps if g[1] < 0]
    chosen, cash_avail, not_executed = [], pf["cash"], []
    for c, gap in gaps:
        if (c, gap) in chosen:
            continue
        if len(chosen) >= max_orders:
            not_executed.append((c, round(gap, 2))); continue
        if gap < 0:
            chosen.append((c, gap)); sells_left.remove((c, gap)); cash_avail += -gap; continue
        while cash_avail < gap and sells_left and len(chosen) < max_orders - 1:
            s = sells_left.pop(0); chosen.append(s); cash_avail += -s[1]
        amount = min(gap, cash_avail)
        if amount < min_trade_eur:
            not_executed.append((c, round(gap, 2))); continue
        chosen.append((c, amount)); cash_avail -= amount
    orders, cash = [], pf["cash"]
    for c, gap in chosen:
        if gap < 0:
            held = pf["units"].get(c, 0.0)
            units = held if target_weights.get(c, 0.0) == 0 else min(held, round(-gap / prices[c], DECIMALS))
            cash += units * prices[c]
            orders.append({"code": c, "action": "sell", "amount_eur": round(units * prices[c], 2),
                           "units": units, "price_T-1": round(prices[c], 4)})
    for c, amount in sorted([x for x in chosen if x[1] > 0], key=lambda x: x[1], reverse=True):
        amount = min(amount, cash)
        units = math.floor(amount / prices[c] * 10**DECIMALS) / 10**DECIMALS
        cash -= units * prices[c]
        orders.append({"code": c, "action": "buy", "amount_eur": math.floor(units * prices[c] * 100) / 100,   # never above the cash
                       "units": units, "price_T-1": round(prices[c], 4)})
    return {"orders": orders, "cash_after": round(cash, 2), "not_executed": not_executed}

def portfolio_after(s):
    """Our portfolio after the approved orders of a run (risk-checked units, cash after)."""
    units = dict(s["portfolio"]["units"])
    for o in s["risk"]["approved"]:
        c = o["code"]
        if o["action"] == "sell" and s.get("weights", {}).get(c, 0) == 0:
            units.pop(c, None); continue                     # full sale: nothing left
        u = units.get(c, 0.0) + (o["units"] if o["action"] == "buy" else -o["units"])
        if u * PRICES[c] >= 0.01:
            units[c] = round(u, DECIMALS)
        else:
            units.pop(c, None)
    return {"cash": round(s["risk"]["cash_after_eur"], 2), "units": units}

def save_portfolio(pf, run_id):
    """Saves trading/portfolio.csv (read by the next run) + a dated copy."""
    df = to_goncalo_portfolio(pf)
    df.to_csv(PORTFOLIO_FILE, index=False)
    df.to_csv(PORTFOLIO_FILE.parent / f"portfolio_after_{run_id}.csv", index=False)
    return df

## 7. Risk rules (Gonçalo) – `check_risk_rules`
Copied unchanged from `Macro_Risk_code/macro_news_risk (revised) (1).ipynb`: hard limits, buys cut to fit (or rejected below €500), sells first, warnings after the trades.

In [ ]:
def check_risk_rules(orders, portfolio=None):
    portfolio = load_portfolio() if portfolio is None else portfolio
    px = last_close
    lev = universe.set_index("code")["leverage"].to_dict()
    holdings = portfolio[portfolio["code"] != "CASH"].set_index("code")["units"].to_dict()
    cash = float(portfolio.loc[portfolio["code"] == "CASH", "units"].sum())
    value = lambda c: holdings.get(c, 0) * px.get(c, 0)
    total = cash + sum(value(c) for c in holdings)          # portfolio value before trading
    approved, rejected = [], []

    for o in orders[MAX_ORDERS_PER_DAY:]:
        rejected.append({**o, "reason": f"more than {MAX_ORDERS_PER_DAY} orders"})
    orders = orders[:MAX_ORDERS_PER_DAY]

    valid = []
    for o in orders:
        c, a, amt = o.get("code"), str(o.get("action", "")).lower(), float(o.get("amount_eur", 0))
        if c not in lev:
            rejected.append({**o, "reason": "ETF not in our list"})
        elif a not in ("buy", "sell"):
            rejected.append({**o, "reason": "action must be buy or sell"})
        elif amt < MIN_ORDER_EUR:
            rejected.append({**o, "reason": f"below {MIN_ORDER_EUR:.0f} EUR"})
        else:
            valid.append({"code": c, "action": a, "amount_eur": amt})

    for o in [o for o in valid if o["action"] == "sell"]:
        held = value(o["code"])
        if held <= 0:
            rejected.append({**o, "reason": "we don't hold it"}); continue
        amt = min(o["amount_eur"], held)
        holdings[o["code"]] -= amt / px[o["code"]]
        cash += amt
        approved.append({**o, "amount_eur": round(amt, 2), "units": round(amt / px[o["code"]], 4),
                         "price_T-1": round(px[o["code"]], 4),
                         "note": "cut to what we hold" if amt < o["amount_eur"] else ""})

    is_lev  = lambda c: lev.get(c, 1) != 1
    is_def  = lambda c: c in DEFENSIVE_ETFS
    theme   = etf_theme
    lev_value   = lambda: sum(value(c) for c in holdings if is_lev(c))
    def_value   = lambda: cash + sum(value(c) for c in holdings if is_def(c))
    theme_value = lambda t: sum(value(c) for c in holdings if theme(c) == t and c not in EXEMPT_FROM_LIMITS)

    for o in [o for o in valid if o["action"] == "buy"]:
        c, amt, why = o["code"], o["amount_eur"], []
        def cap(room, reason):
            nonlocal amt
            if amt > room:
                amt = max(room, 0)
                why.append(reason)
        if c not in EXEMPT_FROM_LIMITS:
            cap(MAX_WEIGHT_PER_ETF * total - value(c), f"max {MAX_WEIGHT_PER_ETF:.0%} per ETF")
            cap(MAX_THEME_WEIGHT * total - theme_value(theme(c)), f"max {MAX_THEME_WEIGHT:.0%} in theme '{theme(c)}'")
        if is_lev(c):
            cap(MAX_LEVERAGED_WEIGHT * total - lev_value(), f"max {MAX_LEVERAGED_WEIGHT:.0%} in leveraged ETFs")
        if not is_def(c):
            cap(def_value() - MIN_DEFENSIVE_WEIGHT * total, f"min {MIN_DEFENSIVE_WEIGHT:.0%} defensive")
        cap(cash - MIN_CASH_EUR, "not enough cash")
        if amt < MIN_ORDER_EUR:
            rejected.append({**o, "reason": "; ".join(why) or "too small"}); continue
        holdings[c] = holdings.get(c, 0) + amt / px[c]
        cash -= amt
        approved.append({**o, "amount_eur": round(amt, 2), "units": round(amt / px[c], 4),
                         "price_T-1": round(px[c], 4), "note": "; ".join(why)})

    weights = {c: float(round(value(c) / total, 3)) for c in holdings if holdings[c] > 1e-9}
    weights["CASH"] = float(round(cash / total, 3))
    summary = {"defensive": float(round(def_value() / total, 3)),
               "uninvested_cash": float(round(cash / total, 3)),
               "leveraged": float(round(lev_value() / total, 3)),
               "themes": {t: float(round(theme_value(t) / total, 3))
                          for t in sorted({theme(c) for c in holdings if holdings[c] > 1e-9 and c not in EXEMPT_FROM_LIMITS})}}
    warnings = [f"{c} is {w:.0%} of the portfolio (max {MAX_WEIGHT_PER_ETF:.0%})"
                for c, w in weights.items() if c != "CASH" and c not in EXEMPT_FROM_LIMITS and w > MAX_WEIGHT_PER_ETF + 0.005]
    warnings += [f"theme '{t}' is {w:.0%} (max {MAX_THEME_WEIGHT:.0%})"
                 for t, w in summary["themes"].items() if w > MAX_THEME_WEIGHT + 0.005]
    if summary["leveraged"] > MAX_LEVERAGED_WEIGHT + 0.005:
        warnings.append(f"leveraged ETFs are {summary['leveraged']:.0%} (max {MAX_LEVERAGED_WEIGHT:.0%})")
    if summary["defensive"] < MIN_DEFENSIVE_WEIGHT - 0.005:
        warnings.append(f"defensive assets are {summary['defensive']:.0%} (min {MIN_DEFENSIVE_WEIGHT:.0%})")
    if summary["uninvested_cash"] > 0.10:
        warnings.append(f"{summary['uninvested_cash']:.0%} is uninvested cash, which earns no interest: "
                        "consider OVERNIGHT CASH")
    return {"approved": approved, "rejected": rejected, "cash_after_eur": round(cash, 2),
            "weights_after": weights, "summary": summary, "warnings": warnings}

## 8. Macro/News agent (Gonçalo)
Copied from the revised notebook: numbered headlines (`IR1`, `GEO3`…), market context (ECB, not Fed, for EUR cash and bonds), **one net view per theme** in JSON, and `macro_conflicts()` for orders that go against the macro view.

Since 16/11 (Gonçalo): bonds are split into **EUR bonds** and **US bonds** for the macro view, so Fed news no longer affects EUR GOVT BOND (the risk limits keep the data themes). `etf_macro_views()` gives the views that matter for one ETF (QUANTUM and SPACE also look at tech, BEL 20 also at European equity, inverse ETFs take the **opposite** view). `macro_view_of()` turns them into one view for the Portfolio Manager, prudently: if one of them is NEGATIVE, that one counts.

In [ ]:
import feedparser
from urllib.parse import quote

NEWS_TOPICS = {
    "interest_rates": ("IR",  "ECB OR Federal Reserve interest rates"),
    "ecb_europe":     ("EU",  "ECB Lagarde euro zone"),
    "geopolitics":    ("GEO", "geopolitical risk markets"),
    "commodities":    ("COM", "oil OR copper OR gold prices"),
}
N_HEADLINES = 8

def get_headlines(query, n=N_HEADLINES):
    url = f"https://news.google.com/rss/search?q={quote(query)}+when:7d&hl=en&gl=US&ceid=US:en"
    try:
        entries = feedparser.parse(url).entries[:n]
    except Exception as e:
        print(f"news unavailable ({type(e).__name__})")
        return []
    items = []
    for e in entries:
        t = e.get("published_parsed")
        items.append((time.strftime("%Y-%m-%d", t) if t else "", e.get("title", "")))
    return sorted(items, reverse=True)

def get_all_headlines():
    out = {}
    for topic, (prefix, query) in NEWS_TOPICS.items():
        for i, (d, title) in enumerate(get_headlines(query), 1):
            out[f"{prefix}{i}"] = f"{title} ({d})"
    return out

# Macro themes = the data themes, except bonds, which are split by the interest rates that drive them.
# (The risk limits keep using the data themes.)
MACRO_THEME_SPLIT = {
    "EUR GOVT BOND":   "EUR bonds",
    "GLOBAL AGG BOND": "EUR bonds",     # hedged to EUR
    "HIGHY BOND":      "EUR bonds",     # EUR high-yield corporate bonds
    "US TREASURY 20Y": "US bonds",
}

def macro_theme(code):
    """Theme used by the macro agent for one ETF."""
    return MACRO_THEME_SPLIT.get(code, THEME_OF.get(code))

THEMES = sorted({macro_theme(c) for c in universe["code"]})
THEME_ETFS = {t: sorted(c for c in universe["code"] if macro_theme(c) == t) for t in THEMES}
VIEWS = ("positive", "neutral", "negative")

MARKET_CONTEXT = """- All our ETFs trade in EUR.
- OVERNIGHT CASH (money market) follows the ECB short-term rate (EUR STR), not the Fed.
- "EUR bonds" (EUR GOVT BOND, GLOBAL AGG BOND hedged to EUR, HIGHY BOND) depend on ECB / euro-area
  interest rates and credit spreads, not on the Fed.
- "US bonds" (US TREASURY 20Y) depend on the Fed and US long-term rates.
- The Fed also matters for US equity, the US dollar and global risk appetite.
- Leveraged ETFs follow their underlying index (e.g. NASDAQ 3X = US tech, 3 times the daily move)."""

def _parse_json(text):
    m = re.search(r"\{.*\}", text or "", re.S)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

def _clean_views(raw):
    views = {t: {"view": "neutral", "reason": "no direct evidence in the headlines", "sources": []} for t in THEMES}
    for v in (raw or {}).get("theme_views", []):
        t, view = v.get("theme"), str(v.get("view", "")).lower()
        if t in views and view in VIEWS:
            views[t] = {"view": view, "reason": v.get("reason", ""), "sources": list(v.get("sources", []))}
    return views

def _with_compat(views):
    """Keeps the old key "Fixed income / Bonds" for code written before the split (copies the EUR bonds view)."""
    v = dict(views)
    eur = views.get("EUR bonds", {"view": "neutral", "reason": "", "sources": []})
    v["Fixed income / Bonds"] = {**eur, "reason": f"same as EUR bonds: {eur.get('reason', '')}"}
    return v

def format_summary(data, views):
    lines = ["FACTS (from the headlines)"]
    lines += [f"- {f.get('text', '')} [{', '.join(f.get('sources', []))}]" for f in data.get("facts", [])]
    lines.append("\nNET VIEW PER THEME (inference)")
    for t, v in views.items():
        if v["sources"]:
            lines.append(f"- {t}: {v['view'].upper()} - {v['reason']} [{', '.join(v['sources'])}]")
    neutral = [t for t, v in views.items() if not v["sources"]]
    if neutral:
        lines.append(f"- No direct evidence (neutral): {', '.join(neutral)}")
    lines.append(f"\nOVERALL\n{data.get('overall', '')}")
    return "\n".join(lines)

def get_macro_news(retries=2, pause=20):
    headlines = get_all_headlines()
    empty = {"facts": [], "overall": "", "theme_views": _with_compat(_clean_views(None)), "headlines": headlines}
    if not headlines:
        return {**empty, "summary": "[no headlines]"}
    llm = get_llm()
    if llm is None:
        return {**empty, "summary": "[AI not configured]"}

    numbered = "\n".join(f"[{h}] {t}" for h, t in headlines.items())
    prompt = f"""You are the macro analyst of a team managing a EUR ETF portfolio for the next 2 weeks.

Headlines from the last 7 days, each with an id:
{numbered}

Market context (always true, use it):
{MARKET_CONTEXT}

Our ETF themes (use these exact names) and the ETFs in each: {json.dumps(THEME_ETFS)}

Rules:
1. Use ONLY the headlines above. Do not invent numbers, dates, names or facts.
2. Cite the headline id(s) next to every fact and every view, e.g. ["IR2"].
3. Do not generalise: if a headline mentions one official or one country, say so.
4. Keep facts (what the headlines say) separate from views (what it means for a theme).
5. Link a headline to a theme only through a direct, well-known mechanism
   (e.g. higher ECB rates -> lower EUR bond prices). If the link is indirect or uncertain, leave the theme out.
6. Give ONE net view per theme, weighing all topics together. If the evidence is mixed or weak, use "neutral".
7. No allocation or trading advice (no buy, sell, overweight, underweight): that is the Portfolio Manager's job.

Answer ONLY with JSON in exactly this format:
{{"facts": [{{"text": "...", "sources": ["IR1"]}}],
 "theme_views": [{{"theme": "<exact theme name>", "view": "positive|neutral|negative", "reason": "<one line>", "sources": ["IR1"]}}],
 "overall": "<2-3 sentences on the macro picture, no allocation advice>"}}"""

    for attempt in range(retries + 1):
        try:
            data = _parse_json(llm.invoke(prompt).content)
            if data:
                views = _clean_views(data)
                return {"facts": data.get("facts", []), "overall": data.get("overall", ""),
                        "theme_views": _with_compat(views), "headlines": headlines, "summary": format_summary(data, views)}
            print("AI answer was not valid JSON, trying again...", flush=True)
        except Exception as e:
            wait = 65 if "RateLimit" in type(e).__name__ else pause      # per-minute quota: wait a full minute
            print(f"AI busy ({type(e).__name__}), waiting {wait}s and trying again...", flush=True)
            time.sleep(wait)
    return {**empty, "summary": "[AI unavailable - see raw headlines]"}

# Which macro views matter for an ETF (+1 = same direction, -1 = opposite, for inverse ETFs) - Gonçalo
MACRO_LOOKUP = {
    "QUANTUM":       [("Frontier tech", +1), ("Equity / Tech", +1)],
    "SPACE":         [("Frontier tech", +1), ("Equity / Tech", +1)],
    "BEL 20":        [("Home market", +1), ("European equity", +1)],
    "NASDAQ 3X":     [("Equity / Tech", +1)],
    "EURO STOXX 2X": [("European equity", +1)],
    "COPPER LVG":    [("Industrial & strategic metals", +1)],
    "PALLADIUM LVG": [("Precious metals", +1)],
    "S&P 500 SHORT": [("US equity", -1)],         # inverse: gains when US equity falls
    "STOXX SHORT":   [("European equity", -1)],   # inverse: gains when European equity falls
}   # every other ETF: its own macro theme, same direction
FLIP = {"positive": "negative", "negative": "positive", "neutral": "neutral"}

def etf_macro_views(code, theme_views):
    """The macro views that matter for one ETF, already turned the right way for inverse ETFs."""
    out = []
    for theme, sign in MACRO_LOOKUP.get(code, [(macro_theme(code), +1)]):
        v = theme_views.get(theme)
        if v:
            out.append({"theme": theme, "view": v["view"] if sign > 0 else FLIP[v["view"]],
                        "reason": v.get("reason", ""), "inverse": sign < 0})
    return out

def macro_view_of(code, theme_views):
    """ONE macro view per ETF for the Portfolio Manager (prudent: if one of its views is negative, that one counts)."""
    views = etf_macro_views(code, theme_views)
    for v in views:
        if v["view"] == "negative":
            return {**v, "reason": f"{v['theme']}: {v['reason']}"}
    return views[0] if views else {}

def macro_conflicts(orders, theme_views):
    """Orders that go against the macro view. Prudent: one opposite view is enough to flag an order."""
    out = []
    for o in orders:
        for v in etf_macro_views(o["code"], theme_views):
            if (o["action"] == "buy" and v["view"] == "negative") or (o["action"] == "sell" and v["view"] == "positive"):
                note = " (inverse ETF, so the opposite of the theme view)" if v["inverse"] else ""
                out.append(f"{o['action'].upper()} {o['code']}: macro view on '{v['theme']}' gives "
                           f"{v['view']}{note} ({v['reason']})")
    return out

## 8b. Technical Analysis agent (Juliet)
Juliet's `technical_agent.py` (branch `juliet_technical`, folder `Technical_code/`) is **imported, not copied**: when she updates her file, upload the new one and the graph uses it. Pure Python, **no AI call**: it works even when Gemini is over its quota.

For every ETF: `tech_score` (momentum: average rank of 6-month and 12-1-month returns, 0 = worst, 1 = best), `tech_view` (positive / neutral / negative, with the 50-day average as a filter), `confidence`, RSI, 20-day breakout and a timing note. Leveraged/inverse ETFs follow their unleveraged twin. Her evidence (2022–2026): momentum is the only indicator with a stable predictive power over 5–10 days; RSI, breakout and MA50 are only timing notes.

Plugged in as in section 8 of her notebook: a `technical` node after `macro`, its summary goes into the Portfolio Manager prompt, and the validation applies her `technical_conflicts`: buying an ETF with a **negative technical view** needs *"Contrary to technical signal because …"*. If the file is missing or fails, the graph runs **without** technical signals (and says so).

In [ ]:
import importlib
sys.path.insert(0, str(Path("Technical_code").resolve()))
try:
    import technical_agent as ta
    ta = importlib.reload(ta)                                  # picks up a newer file uploaded in the same session
    print("Technical agent loaded from", Path(ta.__file__).relative_to(Path.cwd()))
except ImportError:
    ta = None
    print("!! Technical_code/technical_agent.py not found: upload it (section 1) and run this cell again.\n"
          "   Without it the graph still runs, without technical signals.")

def get_technical_signals():
    """Juliet's agent on the same in-memory prices as the other agents. Never stops the graph."""
    empty = {"as_of": "", "summary": None, "theme_views": {}, "signals": []}
    if ta is None:
        return {**empty, "error": "technical_agent.py not loaded"}
    try:
        r = ta.run_technical_agent(prices)
    except Exception as e:
        return {**empty, "error": f"technical agent failed ({type(e).__name__}: {e})"}
    return {"as_of": r["as_of"], "summary": r["summary"], "theme_views": r["theme_views"], "error": None,
            "signals": json.loads(r["signals"].reset_index().to_json(orient="records", date_format="iso"))}

def tech_table(technical):
    """Technical signals as a table indexed by ETF code (empty table if not available)."""
    s = pd.DataFrame((technical or {}).get("signals") or [])
    return s.set_index("code") if len(s) else pd.DataFrame(columns=["tech_view", "reason"])

def tech_view_of(code, technical):
    t = tech_table(technical)
    return t.loc[code, "tech_view"] if code in t.index else "-"

def tech_conflicts_orders(orders, technical):
    """Buy orders on an ETF with a NEGATIVE technical view (shown to the team, like macro_conflicts)."""
    t = tech_table(technical)
    return [f"BUY {o['code']}: technical view is negative ({t.loc[o['code'], 'reason']})"
            for o in orders if o["action"] == "buy" and o["code"] in t.index and t.loc[o["code"], "tech_view"] == "negative"]

_tech = get_technical_signals()
if _tech["error"]:
    print("Technical signals:", _tech["error"])
else:
    _t = tech_table(_tech)
    print(f"Technical signals as of {_tech['as_of']}: {_t['tech_view'].value_counts().to_dict()}")
    print("Negative technical view:", ", ".join(_t.index[_t["tech_view"] == "negative"]) or "none")
    print("Positive technical view:", ", ".join(_t.index[_t["tech_view"] == "positive"]) or "none")

## 9. AI Portfolio Manager (Francesco)
### 9a. Validation of the proposed weights (Python, before any order)
Planning limits (25% defensive, 35% per theme), hard per-ETF and leverage limits, data quality, max 5 orders, the **macro check** (buying an ETF with a NEGATIVE macro view needs "Contrary to macro view because…"), the **technical check** (Juliet: buying an ETF with a NEGATIVE technical view needs "Contrary to technical signal because…") and the **citation check**: every number the AI writes as `name=value` must be the one in its input tables (data table and technical table). Otherwise the proposal goes back to the AI.

In [ ]:
def weights_summary(target_weights):
    """Key figures of a set of target weights (Python), with the same theme logic as check_risk_rules."""
    lev = universe.set_index("code")["leverage"].to_dict()
    invested = sum(target_weights.values())
    cash_left = 1 - invested
    themes = {}
    for c, w in target_weights.items():
        if c not in EXEMPT_FROM_LIMITS:
            themes[etf_theme(c)] = themes.get(etf_theme(c), 0) + w
    return {"invested": round(invested, 4), "cash_left": round(cash_left, 4),
            "defensive": round(cash_left + sum(w for c, w in target_weights.items() if c in DEFENSIVE_ETFS), 4),
            "leveraged": round(sum(w for c, w in target_weights.items() if lev.get(c, 1) != 1), 4),
            "themes": {t: round(w, 4) for t, w in sorted(themes.items(), key=lambda x: -x[1])}}

def moves_needed(pf, target_weights, min_trade_eur=MIN_ORDER_EUR):
    total = portfolio_value(pf)
    etfs = list(target_weights) + sorted(c for c in pf["units"] if c not in target_weights)
    return [c for c in etfs if abs(target_weights.get(c, 0.0) * total - pf["units"].get(c, 0.0) * PRICES[c]) >= min_trade_eur]

CITED = re.compile(r"(ret_1m|ret_3m|ret_6m|vol_3m|max_dd_1y|current_dd|vs_ma50)(?:_%)?\s*[=:]\s*(-?\d+(?:\.\d+)?)")
CITED_TECH = re.compile(r"(tech_score|mom_6m|mom_12_1|rsi14)(?:_%)?\s*[=:]\s*(-?\d+(?:\.\d+)?)")
TECH_TOLERANCE = {"tech_score": 0.0051, "mom_6m": 0.051, "mom_12_1": 0.051, "rsi14": 0.51}
TECH_COLUMN = {"tech_score": "tech_score", "mom_6m": "mom_6m_%", "mom_12_1": "mom_12_1_%", "rsi14": "rsi14"}

def check_cited_numbers(reasons, technical=None):
    """Every 'metric=value' the AI writes in a reason must match its input tables (data + technical)."""
    table, tech, errors = PM_TABLE.set_index("code"), tech_table(technical), []
    for c, reason in reasons.items():
        if c in table.index:
            for name, val in CITED.findall(reason):
                true = table.loc[c, f"{name}_%"]
                if pd.notna(true) and abs(float(val) - float(true)) > 0.051:
                    errors.append(f"{c}: you wrote {name}={val}% but the table says {true}% - copy the numbers from the table")
        if c in tech.index:
            for name, val in CITED_TECH.findall(reason):
                true = tech.loc[c, TECH_COLUMN[name]]
                if pd.notna(true) and abs(float(val) - float(true)) > TECH_TOLERANCE[name]:
                    errors.append(f"{c}: you wrote {name}={val} but the technical table says {true} - copy the numbers from the table")
    return errors

def pre_check_weights(pf, target_weights, reasons=None, theme_views=None, technical=None):
    """List of problems (empty = OK). reasons/theme_views/technical given -> also the macro, technical and
    citation checks (AI proposals). Human edits get the rules only."""
    errors = check_weights(target_weights)
    if errors:
        return errors
    total = portfolio_value(pf)
    flags = PM_TABLE.set_index("code")["data_flag"].to_dict()
    for c, w in target_weights.items():
        current_w = pf["units"].get(c, 0.0) * PRICES[c] / total
        buying = w > current_w + 0.005
        if c not in EXEMPT_FROM_LIMITS and w > MAX_WEIGHT_PER_ETF + 1e-9:
            errors.append(f"{c}: weight {w:.0%} > max {MAX_WEIGHT_PER_ETF:.0%} per ETF")
        if flags.get(c, "ok").startswith("check data") and buying:
            errors.append(f"{c}: data flagged as unreliable ({flags[c]}) - do not buy it")
        if current_w == 0 and 0 < w * total < MIN_ORDER_EUR:
            errors.append(f"{c}: weight {w:.1%} = {w*total:,.0f} EUR, below the min order of {MIN_ORDER_EUR:,.0f} EUR")
        if theme_views is not None and reasons is not None and buying:
            v = macro_view_of(c, theme_views)
            if v.get("view") == "negative" and "contrary to macro view" not in reasons.get(c, "").lower():
                errors.append(f"{c}: its macro view is NEGATIVE ({v.get('reason', '')}) - "
                              "remove it or explain in its reason: 'Contrary to macro view because ...'")
    if reasons is not None:
        if technical is not None and ta is not None and technical.get("signals"):
            errors += ta.technical_conflicts(target_weights, tech_table(technical), reasons, current_weights(pf))
        errors += check_cited_numbers(reasons, technical)
    s = weights_summary(target_weights)
    if s["leveraged"] > MAX_LEVERAGED_WEIGHT + 1e-9:
        errors.append(f"leveraged/inverse ETFs total {s['leveraged']:.0%} > max {MAX_LEVERAGED_WEIGHT:.0%}")
    if s["defensive"] < PLAN_MIN_DEFENSIVE - 1e-9:
        errors.append(f"defensive assets (incl. uninvested cash) only {s['defensive']:.0%} < planning min {PLAN_MIN_DEFENSIVE:.0%}")
    for t, w in s["themes"].items():
        if w > PLAN_MAX_THEME + 1e-9:
            errors.append(f"theme '{t}' totals {w:.0%} > planning max {PLAN_MAX_THEME:.0%}")
    n = len(moves_needed(pf, target_weights))
    if STRICT_MAX_MOVES and n > MAX_ORDERS_PER_DAY:
        errors.append(f"{n} orders needed to reach these weights, max {MAX_ORDERS_PER_DAY} per day - keep only the most important changes")
    return errors

# Quick tests (no AI): expected results in the comments
_pf = new_portfolio()
_tests = {
    "A. valid (expected OK)": {"OVERNIGHT CASH": 0.30, "WORLD MIN VOL": 0.20, "STAPLES": 0.20, "EUR GOVT BOND": 0.15, "WORLD": 0.15},
    "B. tech 45% incl. NASDAQ 3X + QUANTUM (expected: theme 'Equity / Tech' 45% > 35%)":
        {"NASDAQ": 0.20, "NASDAQ 3X": 0.10, "QUANTUM": 0.15, "OVERNIGHT CASH": 0.30},
    "C. on the hard limits (expected: defensive 20% < 25% and 'Size & style factors' 40% > 35%)":
        {"OVERNIGHT CASH": 0.20, "WORLD MIN VOL": 0.20, "STAPLES": 0.20, "HEALTH CARE": 0.20, "DIVIDEND LEADERS": 0.20},
}
for name, w in _tests.items():
    errs = pre_check_weights(_pf, w)
    print(f"{name}\n   -> {'OK' if not errs else errs}")
_wm = PM_TABLE.set_index("code").loc["WORLD MIN VOL", "ret_3m_%"]
print("D. citation check (expected: 1 error, the AI wrote a wrong ret_3m for WORLD MIN VOL)\n   ->",
      check_cited_numbers({"WORLD MIN VOL": f"Data: ret_3m={_wm + 2}% | Macro: none | Judgement: test"}))
_flags = PM_TABLE.set_index("code")["data_flag"]
_tt = tech_table(_tech)
_neg = [c for c in _tt.index[_tt["tech_view"] == "negative"] if _flags.get(c, "ok") == "ok" and c not in UNDERLYING_THEME and c in PRICES]
if _neg:
    _neutral = {t: {"view": "neutral", "reason": "", "sources": []} for t in THEMES}
    _w = {"OVERNIGHT CASH": 0.30, _neg[0]: 0.10}
    print(f"E. technical check (expected: 1 error, buying {_neg[0]} with a NEGATIVE technical view and no explanation)\n   ->",
          pre_check_weights(_pf, _w, {c: "Data: test | Macro: none | Judgement: test" for c in _w}, _neutral, _tech))
    print("   same, with 'Contrary to technical signal because ...' (expected OK) ->",
          pre_check_weights(_pf, _w, {"OVERNIGHT CASH": "test", _neg[0]: "Contrary to technical signal because test"}, _neutral, _tech) or "OK")
    _ts = _tt.loc[_neg[0], "tech_score"]
    print(f"F. technical citation check (expected: 1 error, wrong tech_score for {_neg[0]})\n   ->",
          check_cited_numbers({_neg[0]: f"Data: tech_score={_ts + 0.2:.2f} | Judgement: test"}, _tech))
else:
    print("E/F. technical checks skipped:", _tech.get("error") or "no ETF with a negative technical view today")

### 9b. The prompt (Role / Objective / Inputs / Workflow / Rules / Completion) and the AI call
The data table the AI receives has two columns added at run time: `macro_view` (the view Python will check, from `macro_view_of`) and `exdiv_window` (past ex-dividend dates in the December window). The technical signals (Juliet) come as a second table, with her method in a few lines.

In [ ]:
PM_COLUMNS = ["code", "theme", "exposure_theme", "leverage", "defensive", "ret_1m_%", "ret_3m_%", "ret_6m_%",
              "vol_3m_%", "max_dd_1y_%", "current_dd_%", "vs_ma50_%", "zero_vol_days_1m", "exdiv_window", "data_flag"]

def portfolio_snapshot(pf):
    total = portfolio_value(pf)
    lines = [f"Total value: {total:,.2f} EUR | uninvested cash: {pf['cash']:,.2f} EUR ({pf['cash']/total:.1%})"]
    for c, u in sorted(pf["units"].items(), key=lambda x: -x[1] * PRICES[x[0]]):
        lines.append(f"- {c}: {u * PRICES[c]:,.2f} EUR ({u * PRICES[c] / total:.1%})")
    if not pf["units"]:
        lines.append("- no positions yet (first trading day: everything is in cash)")
    return "\n".join(lines)

def build_pm_prompt(pf, macro, technical_signals=None, feedback=None):
    views = macro.get("theme_views", {})
    t = PM_TABLE[PM_COLUMNS].copy()
    t.insert(3, "macro_view", [macro_view_of(c, views).get("view", "neutral") for c in t["code"]])
    table = t.to_csv(index=False)
    tech_text = technical_signals or "Not available today (the Technical agent did not run): decide without technical signals."
    prompt = f"""ROLE
You are the Portfolio Manager agent of a student team managing a virtual ETF portfolio of about {STARTING_CAPITAL:,.0f} EUR
in the Belfius ReBel competition. Risk profile: BALANCED. You do not execute anything: you propose, humans approve.

OBJECTIVE
Propose the target portfolio weights for the period until the next trading days (competition ends on 23 December 2026),
combining the macro view, the market metrics and the technical signals, and explain every choice.

INPUTS
1) Macro/News view, written by another AI agent from news headlines (facts cite headline ids like [IR2]).
   Treat it critically: it can be wrong.
{macro.get("summary", "[no macro view]")}

2) Market context (always true):
{MARKET_CONTEXT}

3) Market metrics per ETF, computed in Python by the Data agent (% values; vol = annualised volatility over 3 months;
max_dd_1y = worst fall from a peak in 1 year; current_dd = distance from the all-time high; vs_ma50 = price vs its
50-day average; exposure_theme = the theme used for the theme limit: leveraged ETFs count as their underlying,
Frontier tech counts as Equity / Tech; macro_view = the macro view our Python check uses for this ETF;
exdiv_window = dates in past years when this ETF paid a dividend between 5 and 23 December, with the price drop):
{table}
4) Technical signals, computed in Python by the Technical agent (no AI). They are probabilities, not certainties:
   in our backtest a pure momentum strategy beat WORLD only about half of the time.
{tech_text}

5) Current portfolio:
{portfolio_snapshot(pf)}

WORKFLOW
1. Read the macro view and decide which themes to overweight, keep neutral or avoid. Separate facts from interpretations.
2. Check the metrics: trend (ret_1m, ret_3m, vs_ma50), risk (vol_3m, max_dd_1y), data quality (data_flag) and dividends (exdiv_window).
3. Check the technical signals: tech_view and tech_score (momentum) are the main technical evidence; rsi14, breakout_20d
   and timing_note are weak signals, use them only to time an entry. Note where the macro and technical views disagree.
4. Choose the ETFs and their weights, listed in ORDER OF PRIORITY (most important first).
5. Check every rule below before answering.
6. Write a short investment memo.

RULES (checked afterwards by Python - a proposal that breaks them is rejected and sent back to you)
- Weights are fractions of the total portfolio (0.20 = 20%), use multiples of 0.05. Their sum must be <= 1; the rest stays as uninvested cash.
- Your weights are the COMPLETE target portfolio: any ETF we hold that you leave out will be SOLD.
- Max {MAX_WEIGHT_PER_ETF:.0%} in a single ETF, except OVERNIGHT CASH (money market fund, behaves like cash).
- Max {MAX_LEVERAGED_WEIGHT:.0%} in total in leveraged/inverse ETFs (leverage column different from 1).
- At least {PLAN_MIN_DEFENSIVE:.0%} in defensive assets = uninvested cash + {", ".join(DEFENSIVE_ETFS)} (column defensive = True).
  (Our hard limit is {MIN_DEFENSIVE_WEIGHT:.0%}: we plan with a safety margin because prices move between our trading days and we can rebalance only on 5 days.)
  WARNING: the theme called "Defensive sector" is NOT defensive for this rule (DEFENSE is a military/aerospace equity ETF).
- Max {PLAN_MAX_THEME:.0%} in a single exposure_theme (OVERNIGHT CASH not counted; hard limit {MAX_THEME_WEIGHT:.0%}, same safety margin).
- Every new position must be at least {MIN_ORDER_EUR:,.0f} EUR (weight >= {MIN_ORDER_EUR/STARTING_CAPITAL:.0%} at the current size).
- Max {MAX_ORDERS_PER_DAY} orders per trading day (buys and sells both count): the changes versus the current portfolio must need at most {MAX_ORDERS_PER_DAY} orders.
- Do not buy ETFs whose data_flag starts with "check data" (unreliable prices).
- Dividends are NOT credited in this competition (portfolios are valued at closing prices only). If an ETF has dates in
  exdiv_window, holding it on that day in December would probably cost us that price drop; the ret_* columns include
  dividends, so they overstate what we would earn on these ETFs. Say it in the reason if you choose such an ETF.
- Under "Data:" write the metrics as name=value copied from the tables (e.g. ret_3m=9.5%, tech_score=0.71, tech_view=positive):
  Python checks every number.
- Under "Macro:" write ONLY points that are actually in the macro view above (cite the headline ids); general finance knowledge goes under "Judgement:".
- If you give weight to an ETF whose macro_view is negative, its reason MUST contain "Contrary to macro view because ...".
- If you give weight to an ETF whose tech_view is negative, its reason MUST contain "Contrary to technical signal because ...".
- Use ONLY the numbers given above. Do not compute new statistics and do not invent facts, numbers or news.
- Do not chase extreme past returns: for a balanced profile, high volatility needs a strong, explicit reason.

COMPLETION
Answer with ONLY one valid JSON object, no text before or after, in exactly this format:
{{"weights": [{{"code": "ETF code exactly as in the table", "weight": 0.20,
               "reason": "Data: <metrics used, as name=value> | Macro: <macro point used, with ids, or none> | Judgement: <your reasoning>"}}],
 "memo": "Investment memo, max 250 words: 1) market view, 2) main choices, 3) how the portfolio differs from 100% WORLD (our benchmark), 4) key risks, 5) what would make us change, 6) themes with a POSITIVE macro view that we did not use, and why, 7) ETFs where the macro and technical views disagree, and how you weighed them."}}
"""
    if feedback:
        prompt += f"""
CORRECTION NEEDED
Your previous proposal was rejected by our Python checks:
{feedback}
Fix ALL these problems and return the complete JSON again."""
    return prompt

def parse_pm_json(text):
    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("no JSON object found in the answer")
    data = json.loads(text[start:end + 1])
    if not isinstance(data.get("weights"), list) or not data["weights"]:
        raise ValueError("'weights' must be a non-empty list")
    weights, reasons = {}, {}
    for item in data["weights"]:
        c, w = str(item.get("code", "")).strip(), float(item.get("weight", 0))
        if c in weights:
            raise ValueError(f"{c} appears twice")
        if w > 0:
            weights[c] = round(w, 4)
            reasons[c] = str(item.get("reason", ""))
    return weights, reasons, str(data.get("memo", ""))

def ask_portfolio_manager(pf, macro, technical_signals=None, feedback=None, retries=2, pause=20):
    """One call to the AI Portfolio Manager. Always returns a dict; 'error' is None if the answer was usable."""
    prompt = build_pm_prompt(pf, macro, technical_signals, feedback)
    out = {"weights": {}, "reasons": {}, "memo": "", "raw": "", "prompt": prompt, "error": None}
    llm = get_llm()
    if llm is None:
        out["error"] = "AI not configured"; return out
    for attempt in range(retries + 1):
        try:
            out["raw"] = llm.invoke(prompt).content
            break
        except Exception as e:
            wait = 65 if "RateLimit" in type(e).__name__ else pause
            print(f"AI busy ({type(e).__name__}), waiting {wait}s...", flush=True); time.sleep(wait)
    else:
        out["error"] = "AI unavailable"; return out
    try:
        out["weights"], out["reasons"], out["memo"] = parse_pm_json(out["raw"])
    except Exception as e:
        out["error"] = f"answer not in the required JSON format ({e})"
    return out

### 9c. What the team sees before deciding, and the logs
`show_proposal()` prints the AI proposal **and** a preview of the orders and of the risk check, so the team approves the real orders, not only the weights. Next to every ETF: the macro view and the technical view that Python checks. The logs record which model wrote the proposal, both views and who decided.

In [ ]:
def show_proposal(s):
    pf = s["portfolio"]
    warn = prices_warning(s.get("data_summary", {}).get("date_range", ["", today_paris()])[1])
    if warn:
        print(warn + "\n")
    tech = s.get("technical", {})
    if tech.get("error"):
        print(f"!! No technical signals in this run: {tech['error']}\n")
    print(f"========== PROPOSAL ({s.get('source')}, {s.get('pm_attempts')} attempt(s)) ==========")
    for h in s.get("pm_history", []):
        print(f"  attempt {h['attempt']}: {h['weights']} -> {'OK' if not h['errors'] else h['errors']}")
    print("\nTarget weights (priority order), macro and technical views checked by Python, AI reason:")
    views = s.get("macro", {}).get("theme_views", {})
    divs = PM_TABLE.set_index("code")["exdiv_window"].to_dict()
    for c, w in s.get("weights", {}).items():
        print(f"{w:>5.0%}  {c:<16} [macro: {macro_view_of(c, views).get('view', '-')} | tech: {tech_view_of(c, tech)}]  "
              f"{s.get('reasons', {}).get(c, '')}")
        if divs.get(c, "-") != "-":
            print(f"       !! past ex-dividend dates in the window (dividend NOT credited): {divs[c]}")
    print("\nSummary (Python):", weights_summary(s.get("weights", {})))
    print("\nMEMO:\n", s.get("memo", ""))
    prev = weights_to_orders(pf, s.get("weights", {}))
    risk = check_risk_rules(prev["orders"], to_goncalo_portfolio(pf))
    print("\n---------- ORDERS THAT WOULD BE SENT (preview) ----------")
    for o in risk["approved"]:
        note = f"  ({o['note']})" if o.get("note") else ""
        print(f"  {o['action'].upper():4} {o['code']:<16} {o['amount_eur']:>9,.2f} EUR{note}")
    for o in risk["rejected"]:
        print(f"  REJECTED {o['action'].upper():4} {o['code']:<16} -> {o['reason']}")
    print("  Not executed today:", prev["not_executed"] or "none")
    print("  Risk warnings:", risk["warnings"] or "none")
    print("  Orders against the macro view:", macro_conflicts(prev["orders"], views) or "none")
    print("  Orders against the technical view:", tech_conflicts_orders(prev["orders"], tech) or "none")
    if s.get("approval_errors"):
        print("\n!! Your last decision was not valid:", s["approval_errors"])
    print("\nNext: write the team's decision in the next cell (approve / edit / reject) and run it.")

LOG_COLUMNS = ["run_id", "logged_at", "prices_date", "llm_model", "pm_source", "pm_attempts", "team_decision", "decided_by",
               "team_comment", "code", "action", "ai_weight", "final_weight", "macro_view", "tech_view", "ai_reason",
               "risk_outcome", "risk_note", "amount_eur", "units", "price_T-1"]

def write_logs(s):
    """One row per order in trading/trade_log.csv + the full run (prompts, AI answers, decision) in trading/runs/."""
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    views = s.get("macro", {}).get("theme_views", {})
    tech = s.get("technical", {})
    appr = s.get("approval", {})
    model = os.environ.get("LLM_MODEL") or "none"                 # which Gemini model wrote the proposal
    base = {"run_id": s["run_id"], "logged_at": now_paris().strftime("%Y-%m-%d %H:%M"),
            "prices_date": s.get("data_summary", {}).get("date_range", ["", ""])[1], "llm_model": model,
            "pm_source": s.get("source"), "pm_attempts": s.get("pm_attempts"),
            "team_decision": appr.get("action"), "decided_by": appr.get("by", ""), "team_comment": appr.get("comment", "")}
    rows = []
    for o in s["risk"]["approved"] + s["risk"]["rejected"]:
        c = o["code"]
        rows.append({**base, "code": c, "action": o["action"],
                     "ai_weight": s.get("ai_weights", {}).get(c, 0), "final_weight": s.get("weights", {}).get(c, 0),
                     "macro_view": macro_view_of(c, views).get("view", ""), "tech_view": tech_view_of(c, tech),
                     "ai_reason": s.get("reasons", {}).get(c, ""),
                     "risk_outcome": "approved" if "reason" not in o else "rejected",
                     "risk_note": o.get("note", o.get("reason", "")), "amount_eur": o.get("amount_eur"),
                     "units": o.get("units"), "price_T-1": o.get("price_T-1")})
    traded = {r["code"] for r in rows}
    for c, w in s.get("ai_weights", {}).items():          # proposed by the AI but not traded (e.g. removed by the team)
        if c not in traded:
            rows.append({**base, "code": c, "action": "not traded", "ai_weight": w,
                         "final_weight": s.get("weights", {}).get(c, 0), "macro_view": macro_view_of(c, views).get("view", ""),
                         "tech_view": tech_view_of(c, tech), "ai_reason": s.get("reasons", {}).get(c, "")})
    if not rows:
        rows.append({**base, "code": "", "action": "none"})
    if TRADE_LOG_FILE.exists():                           # old log with other columns: realign it once
        old = pd.read_csv(TRADE_LOG_FILE)
        if list(old.columns) != LOG_COLUMNS:
            old.reindex(columns=LOG_COLUMNS).to_csv(TRADE_LOG_FILE, index=False)
    log = pd.DataFrame(rows).reindex(columns=LOG_COLUMNS)         # always the same columns, in the same order
    log.to_csv(TRADE_LOG_FILE, mode="a", header=not TRADE_LOG_FILE.exists(), index=False)
    keep = {k: v for k, v in s.items() if k != "portfolio"} | {"portfolio_before": s["portfolio"], "llm_model": model}
    (RUNS_DIR / f"{s['run_id']}.json").write_text(json.dumps(keep, default=str, indent=1), encoding="utf-8")
    return log

## 10. The team graph
- **interrupt**: the `human_approval` node *pauses* the graph and waits for the team's decision. A **checkpointer** (`MemorySaver`) keeps the paused state in memory, so the graph can resume exactly where it stopped.
- **Conditional edges** (the forks): after `validate` (OK → approval, errors → back to the AI or fallback) and after `human_approval` (invalid edit → ask again).

In [ ]:
from typing import TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

def native(x):
    """numpy numbers -> plain Python numbers (the checkpointer can only save plain types)."""
    return json.loads(json.dumps(x, default=lambda v: v.item() if hasattr(v, "item") else str(v)))

class State(TypedDict, total=False):
    run_id: str
    data_report: str          # data agent
    data_summary: dict
    portfolio: dict
    macro: dict               # macro agent: facts, theme_views, summary, headlines
    technical: dict           # technical agent (Juliet): as_of, summary, theme_views, signals, error
    weights: dict             # current target weights (AI, then possibly edited by the team)
    ai_weights: dict          # what the AI proposed (kept for the log)
    reasons: dict
    memo: str
    pm_attempts: int
    pm_error: str
    pm_errors: list
    pm_history: list          # every attempt: weights, errors, prompt, raw AI answer
    pm_prompt: str
    pm_raw: str
    source: str               # "AI", "fallback", "... + team edit"
    approval: dict            # the team's decision
    approval_errors: list
    orders: list
    not_executed: list
    risk: dict
    macro_conflicts: list
    tech_conflicts: list

def data_node(state):
    print("0) Data: prices, quality checks, metrics...", flush=True)
    report = ""
    if REFRESH_DATA:
        try:
            report = run_data_agent(use_ai=USE_AI_DATA_REPORT)
        except Exception as e:
            report = f"Data refresh failed ({type(e).__name__}: {e}) - using the existing data/prices.csv"
            print("  ", report)
    summary = load_market_data(verbose=False)
    print(f"   prices up to {summary['date_range'][1]} | issues: {summary['issues_by_type']}", flush=True)
    warn = prices_warning(summary["date_range"][1])
    if warn:
        print("  ", warn, flush=True)
    return {"data_report": report, "data_summary": native(summary)}

def macro_node(state):
    print("1) Macro/News: headlines + AI view per theme...", flush=True)
    return {"macro": get_macro_news()}

def technical_node(state):
    print("1b) Technical: momentum, trend, breakout, RSI (Python, no AI)...", flush=True)
    r = get_technical_signals()
    if r.get("error"):
        print(f"   !! {r['error']} -> the Portfolio Manager decides without technical signals", flush=True)
    else:
        counts = pd.Series([x["tech_view"] for x in r["signals"]]).value_counts().to_dict()
        print(f"   signals as of {r['as_of']}: {counts}", flush=True)
    return {"technical": native(r)}

def portfolio_manager_node(state):
    attempt = state.get("pm_attempts", 0) + 1
    print(f"2) Portfolio Manager (AI): proposal, attempt {attempt}/{MAX_PM_ATTEMPTS}...", flush=True)
    feedback = None
    if attempt > 1:
        feedback = f"Previous weights: {state.get('weights', {})}\n" + "\n".join("- " + e for e in state.get("pm_errors", []))
    r = ask_portfolio_manager(state["portfolio"], state["macro"],
                              technical_signals=state.get("technical", {}).get("summary"), feedback=feedback)
    return {"weights": r["weights"], "ai_weights": r["weights"], "reasons": r["reasons"], "memo": r["memo"],
            "pm_attempts": attempt, "pm_error": r["error"], "source": "AI", "pm_prompt": r["prompt"], "pm_raw": r["raw"]}

def validate_node(state):
    print("   Validate: Python checks the proposal...", flush=True)
    if state.get("pm_error"):
        errors = [state["pm_error"]]
    else:
        errors = pre_check_weights(state["portfolio"], state["weights"], state["reasons"], state["macro"]["theme_views"],
                                   state.get("technical"))
    print("   ->", "OK" if not errors else f"{len(errors)} problem(s): " + " | ".join(errors), flush=True)
    entry = {"attempt": state["pm_attempts"], "weights": state.get("weights", {}), "errors": errors,
             "prompt": state.get("pm_prompt", ""), "raw_answer": state.get("pm_raw", "")}
    return {"pm_errors": errors, "pm_history": state.get("pm_history", []) + [entry]}

def route_after_validate(state):
    if not state["pm_errors"]:
        return "human_approval"
    if state.get("pm_error") in ("AI unavailable", "AI not configured") or state["pm_attempts"] >= MAX_PM_ATTEMPTS:
        return "fallback"
    return "portfolio_manager"

def fallback_node(state):
    print("   FALLBACK: no valid AI proposal -> keep the current portfolio. The team decides.", flush=True)
    return {"weights": native(current_weights(state["portfolio"])), "reasons": {}, "source": "fallback",
            "memo": f"FALLBACK after {state['pm_attempts']} attempt(s). Last problems: {state['pm_errors']}. No trades proposed."}

def human_approval_node(state):
    decision = interrupt({"question": "approve / edit / reject?", "errors": state.get("approval_errors", [])})
    if not isinstance(decision, dict):                     # e.g. the word "approve" instead of a dict
        decision = {"action": str(decision)}
    action = str(decision.get("action", "")).lower()
    pf = state["portfolio"]
    if action == "approve":
        return {"approval": decision, "approval_errors": []}
    if action == "edit":
        w = {c: float(x) for c, x in decision.get("weights", {}).items() if float(x) > 0}
        errors = pre_check_weights(pf, w)                      # same rules for humans (no macro / technical check)
        if errors:
            return {"approval": decision, "approval_errors": errors}
        return {"weights": w, "source": f"{state.get('source')} + team edit", "approval": decision, "approval_errors": []}
    if action == "reject":
        return {"weights": native(current_weights(pf)), "approval": decision, "approval_errors": []}
    return {"approval": decision, "approval_errors": [f"unknown action '{action}': use approve, edit or reject"]}

def route_after_approval(state):
    return "human_approval" if state.get("approval_errors") else "orders"

def orders_node(state):
    print("3) Orders: weights -> max 5 feasible orders...", flush=True)
    res = weights_to_orders(state["portfolio"], state["weights"])
    return native({"orders": res["orders"], "not_executed": res["not_executed"],
                   "macro_conflicts": macro_conflicts(res["orders"], state["macro"]["theme_views"]),
                   "tech_conflicts": tech_conflicts_orders(res["orders"], state.get("technical"))})

def risk_node(state):
    print("4) Risk: final check on the hard limits...", flush=True)
    return {"risk": native(check_risk_rules(state["orders"], to_goncalo_portfolio(state["portfolio"])))}

def log_node(state):
    if LOG_RUNS:
        write_logs(state)
        print(f"5) Log: {TRADE_LOG_FILE} and {RUNS_DIR}/{state['run_id']}.json written", flush=True)
    return {}

g = StateGraph(State)
for name, fn in [("data", data_node), ("macro", macro_node), ("technical", technical_node),
                 ("portfolio_manager", portfolio_manager_node), ("validate", validate_node), ("fallback", fallback_node),
                 ("human_approval", human_approval_node), ("orders", orders_node), ("risk", risk_node), ("log", log_node)]:
    g.add_node(name, fn)
g.add_edge(START, "data")
g.add_edge("data", "macro")
g.add_edge("macro", "technical")
g.add_edge("technical", "portfolio_manager")
g.add_edge("portfolio_manager", "validate")
g.add_conditional_edges("validate", route_after_validate,
                        {"human_approval": "human_approval", "portfolio_manager": "portfolio_manager", "fallback": "fallback"})
g.add_edge("fallback", "human_approval")
g.add_conditional_edges("human_approval", route_after_approval, {"human_approval": "human_approval", "orders": "orders"})
g.add_edge("orders", "risk")
g.add_edge("risk", "log")
g.add_edge("log", END)
app = g.compile(checkpointer=MemorySaver())

def start_run(portfolio=None, run_id=None):
    """Runs the graph until the human approval. Returns the 'config' that identifies this run."""
    run_id = run_id or now_paris().strftime("%Y%m%d-%H%M%S")
    config = {"configurable": {"thread_id": run_id}}
    app.invoke({"run_id": run_id, "portfolio": portfolio or new_portfolio()}, config)
    return config

def run_state(config):
    return app.get_state(config).values

def decide(config, decision):
    """Gives the team's decision to the paused graph and lets it finish (or ask again if the edit is invalid)."""
    if not app.get_state(config).next:
        print("This run is already finished: start a new run (section 12) for a new decision.")
        return run_state(config)
    app.invoke(Command(resume=decision), config)
    s = app.get_state(config)
    if s.next:                                            # still waiting: the decision was not valid
        show_proposal(s.values)
    return s.values

print(app.get_graph().draw_mermaid())

## 11. Test WITHOUT Gemini (fake AI, fake macro and technical views, no download, no logs)
Checks the whole flow – correction loop (macro **and** technical check), human approval, an invalid team edit, orders, risk – and then a **second trading day** that starts from the portfolio of the first one (with sells), without using the free Gemini quota. Nothing is written to `trading/`. The technical views are fake here (all neutral, HEALTH CARE negative), so the test gives the same result whatever the market does.

In [ ]:
_real = (ask_portfolio_manager, get_macro_news, get_technical_signals, REFRESH_DATA, LOG_RUNS)

def _fake_ai(answers):
    answers = list(answers); last = answers[-1]
    def fake(pf, macro, technical_signals=None, feedback=None):
        print("   (fake AI) feedback:", "none" if feedback is None else feedback.replace("\n", " | ")[:220])
        w = answers.pop(0) if answers else last
        return {"weights": w, "reasons": {c: "test" for c in w}, "memo": "fake memo", "raw": "", "prompt": "", "error": None}
    return fake

_fake_macro = {"summary": "[test]", "headlines": {}, "facts": [], "overall": "",
               "theme_views": {t: {"view": "neutral", "reason": "", "sources": []} for t in THEMES}}
_fake_macro["theme_views"]["Health care"] = {"view": "negative", "reason": "test negative view", "sources": ["GEO1"]}
_fake_tech = {"as_of": "test", "summary": "[test technical signals]", "theme_views": {}, "error": None,
              "signals": [{"code": c, "tech_view": "negative" if c == "HEALTH CARE" else "neutral", "tech_score": 0.5,
                           "reason": "test negative technical view" if c == "HEALTH CARE" else "test"} for c in universe["code"]]}

BAD  = {"OVERNIGHT CASH": 0.30, "HEALTH CARE": 0.20, "WORLD MIN VOL": 0.20, "STAPLES": 0.15, "WORLD": 0.15}   # health care: macro AND technical negative, no explanation
GOOD = {"OVERNIGHT CASH": 0.30, "WORLD MIN VOL": 0.20, "STAPLES": 0.20, "EUR GOVT BOND": 0.15, "WORLD": 0.15}
DAY2 = {"OVERNIGHT CASH": 0.30, "WORLD MIN VOL": 0.20, "EUR GOVT BOND": 0.15, "WORLD": 0.15, "EUROPE 600": 0.20}  # sell STAPLES, buy EUROPE 600
try:
    ask_portfolio_manager, get_macro_news, get_technical_signals = _fake_ai([BAD, GOOD]), (lambda: _fake_macro), (lambda: _fake_tech)
    REFRESH_DATA, LOG_RUNS = False, False
    _id = now_paris().strftime("%H%M%S")                  # new id every time: a re-run starts clean
    cfg = start_run(run_id=f"TEST-{_id}")
    show_proposal(run_state(cfg))
    print("\n>>> Team tries an INVALID edit (WORLD 30%): the graph must ask again")
    decide(cfg, {"action": "edit", "weights": {"WORLD": 0.30, "OVERNIGHT CASH": 0.70}, "by": "test", "comment": "test"})
    print("\n>>> Team approves")
    final = decide(cfg, {"action": "approve", "by": "test", "comment": "test approval"})
    print("\nApproved by risk:", [(o["code"], o["amount_eur"]) for o in final["risk"]["approved"]])
    print("Rejected:", final["risk"]["rejected"], "| warnings:", final["risk"]["warnings"])

    print("\n>>> SECOND TRADING DAY: start from the portfolio after day 1 (not saved to disk)")
    pf_day2 = portfolio_after(final)
    print("Portfolio after day 1:", {c: round(u, 2) for c, u in pf_day2["units"].items()}, "| cash", pf_day2["cash"])
    ask_portfolio_manager = _fake_ai([DAY2])
    cfg2 = start_run(portfolio=pf_day2, run_id=f"TEST2-{_id}")
    final2 = decide(cfg2, {"action": "approve", "by": "test", "comment": "test day 2"})
    print("Day 2 orders approved by risk:", [(o["action"], o["code"], o["amount_eur"]) for o in final2["risk"]["approved"]])
    print("Portfolio after day 2:", {c: round(u, 2) for c, u in portfolio_after(final2)["units"].items()})
finally:
    ask_portfolio_manager, get_macro_news, get_technical_signals, REFRESH_DATA, LOG_RUNS = _real

## 12. Real run – step 1: data → macro → technical → AI → validation, then STOP for the team
Uses Gemini: 1 macro call + 1 to 3 Portfolio Manager calls (the technical agent uses no AI). Wait a few minutes between runs (free quota).
On a **real trading day** set `REAL_TRADING_DAY = True`: the run starts from `trading/portfolio.csv` (our real portfolio, uploaded in section 1). Otherwise it is a dry run from 100% cash.

In [ ]:
REAL_TRADING_DAY = False

if REAL_TRADING_DAY:
    if not PORTFOLIO_FILE.exists():
        print("!! trading/portfolio.csv not found: on the first trading day (7 Dec) this is normal (100% cash);"
              " on later days upload the portfolio.csv of the previous trading day first.")
    pf_start = from_goncalo_portfolio(load_portfolio())
else:
    pf_start = new_portfolio()
print("Starting portfolio:", pf_start)
cfg = start_run(portfolio=pf_start)
s = run_state(cfg)
print("\n========== DATA ==========\n", s["data_report"][:600])
print("\n========== MACRO VIEW ==========\n", s["macro"]["summary"])
print("\n========== TECHNICAL VIEW PER THEME ==========")
if s["technical"].get("error"):
    print(s["technical"]["error"])
for t, v in s["technical"].get("theme_views", {}).items():
    print(f"- {t}: {v['view']} ({v['reason']})")
print()
show_proposal(s)

## 13. Real run – step 2: the team's decision
Choose **one** decision, write who decided and why (it goes in the trade log), then run the cell. Nothing is approved by default.

In [ ]:
# Choose ONE decision: remove the # in front of it, write who decided (by) and why (comment), then run the cell.
decision = None
# decision = {"action": "approve", "by": "NAME", "comment": "WHY we approve"}
# decision = {"action": "edit", "by": "NAME", "comment": "WHY we change it",
#             "weights": {"OVERNIGHT CASH": 0.30, "WORLD": 0.20, "STAPLES": 0.20, "EUR GOVT BOND": 0.15, "HEALTH CARE": 0.15}}
# decision = {"action": "reject", "by": "NAME", "comment": "WHY we reject (no trades today)"}

if decision is None:
    raise RuntimeError("No decision chosen: remove the # in front of ONE decision above, write 'by' and 'comment', run again.")
if decision.get("by") in (None, "", "NAME") or str(decision.get("comment", "")).startswith("WHY"):
    raise RuntimeError("Write who decided ('by') and the real reason ('comment') before running.")

final = decide(cfg, decision)
if "risk" in final:
    print("\n========== FINAL ORDERS (after risk check) ==========")
    for o in final["risk"]["approved"]:
        note = f"  ({o['note']})" if o.get("note") else ""
        print(f"APPROVED  {o['action'].upper():4} {o['code']:<16} {o['amount_eur']:>9,.2f} EUR  {o['units']} units @ {o['price_T-1']}{note}")
    for o in final["risk"]["rejected"]:
        print(f"REJECTED  {o['action'].upper():4} {o['code']:<16} -> {o['reason']}")
    print("Warnings:", final["risk"]["warnings"] or "none")
    print("Against the macro view:", final["macro_conflicts"] or "none")
    print("Against the technical view:", final.get("tech_conflicts") or "none")
    print(f"Cash after: {final['risk']['cash_after_eur']:,.2f} EUR | summary: {final['risk']['summary']}")

## 13b. Real trading day only: save our new portfolio
Run this **only after** the approved orders have been sent to Belfius. It writes `trading/portfolio.csv`, which the next trading day starts from. When Belfius publishes the leaderboard, compare: if their numbers differ, theirs are right – correct `portfolio.csv` by hand.

In [ ]:
SAVE_PORTFOLIO = False          # True only on a real trading day, after sending the orders to Belfius

if SAVE_PORTFOLIO:
    if not REAL_TRADING_DAY:
        raise RuntimeError("This was a dry run (REAL_TRADING_DAY = False): the portfolio is not saved.")
    print(save_portfolio(portfolio_after(final), final["run_id"]).to_string(index=False))
    print(f"Saved {PORTFOLIO_FILE} - download the logs in section 14 and keep the zip for the next trading day.")
else:
    print("Not saved (SAVE_PORTFOLIO = False).")

## 14. Download the logs (Colab deletes files at the end of the session)

In [ ]:
shutil.make_archive("trading_logs", "zip", "trading")
try:
    from google.colab import files; files.download("trading_logs.zip")
except Exception:
    print("Saved trading_logs.zip in", Path.cwd())